# AdventureWorks CRM 마케팅 비용 최적화

Excel 데이터 로드, 전처리, EDA, RFM 분석, RandomForest 모델 학습 순서로 실행합니다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns

from src.load_data import load_and_merge
from src.preprocessing import clean_sales_data, build_customer_features, build_sales_features


## 1. 데이터 로드 및 품질 확인

In [ ]:
raw_data = load_and_merge()
print('데이터 크기:', raw_data.shape)
print('중복 행 수:', raw_data.duplicated().sum())
display(raw_data.isna().sum().sort_values(ascending=False).head(15))
display(raw_data.head())

In [ ]:
sales = clean_sales_data(raw_data)
display(sales[['order_date_clean', 'sales_amount_clean', 'order_quantity_clean']].describe())

## 2. 월별 매출 분석

In [ ]:
monthly_sales = sales.groupby('year_month', as_index=False)['sales_amount_clean'].sum()

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_sales, x='year_month', y='sales_amount_clean', marker='o')
plt.xticks(rotation=45)
plt.title('Monthly Sales')
plt.tight_layout()
plt.show()

## 3. 지역별 / 상품 카테고리별 매출 분석

In [ ]:
for column, title in [('region', 'Sales by Region'), ('category', 'Sales by Category')]:
    if column in sales.columns:
        summary = (
            sales.groupby(column, as_index=False)['sales_amount_clean']
            .sum()
            .sort_values('sales_amount_clean', ascending=False)
        )
        plt.figure(figsize=(10, 5))
        sns.barplot(data=summary, x='sales_amount_clean', y=column)
        plt.title(title)
        plt.tight_layout()
        plt.show()
    else:
        print(f'{column} 컬럼이 없어 해당 그래프를 건너뜁니다.')

## 4. 고객별 구매금액 및 RFM 분석

In [ ]:
customer_features, cutoff_date = build_customer_features(sales)
print('구매 예측 기준일:', cutoff_date.date())
display(customer_features.head())
display(customer_features[['recency', 'frequency', 'monetary']].describe())

plt.figure(figsize=(8, 5))
sns.histplot(customer_features['monetary'], bins=30)
plt.title('Customer Monetary Distribution')
plt.tight_layout()
plt.show()

## 5. 회귀 모델용 월별 집계 데이터 확인

In [ ]:
sales_features = build_sales_features(sales)
display(sales_features.head())
display(sales_features.describe())

## 6. 모델 학습

아래 셀을 실행하면 모델 파일과 평가 그래프가 `models/`, `outputs/figures/` 폴더에 저장됩니다.

In [ ]:
from src.train_classifier import train_classifier
from src.train_regressor import train_regressor

classifier = train_classifier()
regressor = train_regressor()